In [1]:
import pandas as pd
import re
import unicodedata

In [2]:
df = pd.read_excel('../data/raw/Base Jujuy Consolidada Fer .xlsx',sheet_name='Base_Consolidada',header=1)

In [3]:
df_j = df.copy()

In [6]:
df_j["Actores participantes"].unique().tolist()

['Partidos políticos/Multisectorial de mujere/Comunidades indígenas/organizaciones sociales/ sociedad civil/ Gremios/ Sindicatos | partidos políticos, Multisectorial de mujeres, comunidades indígenas, movimientos sociales, sociedad civil, gremios, sindicatos',
 'Sindicato de Obreros y Empleados del Azúcar de Ingenio Ledesma (Soeail)',
 'Gremios de trabajadores estatales, CTA',
 'Docentes/Gremios  | docentes, gremios',
 'Docentes/Gremios/ Estudiantes | docentes, estudiantes, gremios estatales',
 'docentes (Cedems, Adep, UDA, Sadop), estudiantes, familias, gremios estatales (Seom, Apuap, ATE)',
 'docentes (Adep, Cedems, UDA, Sadop, Asdea, Amet), gremios estatales (Seom, Apuap, ATE)',
 'Docentes/Gremios ',
 'docentes, gremios estatales',
 'docentes (Adep, Cedems)',
 'docentes y gremios locales',
 'Docentes/Gremios/ Organizaciones sociales',
 'docentes',
 'docentes (Adep, Cedems, Sadop, Adiunju), gremios (CTA, Apuap), sociedad civil',
 'Docentes/Gremios/Comunidades indígenas',
 'Organizaci

In [7]:
col_actores = "Actores participantes"

In [8]:
def normalizar(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto

df_j["_actores_norm"] = df_j[col_actores].apply(normalizar)

In [9]:
REGLAS = {
    "Docentes": r"docente|adep|cedems|uda|sadop|adiunju|amet|asdea|intergremial docente|gremio[s]? docente[s]?|comunidad educativa|estudiante|auxiliar",
    "Gremios_estatales": r"estatal|ate\b|seom|apuap|cta\b|trabajador(es)? estatal|empleado publico|intersindical",
    "Sindicatos_privados": r"soeail|azucar|aoma|minero[s]?|trabajador(es)? minero|sindicato de obrero",
    "Organizaciones_sociales": r"organizacion(es)? social|movimiento social|piqueter|mte|multisectorial|partido obrero|frente de izq|dirigente[s]? social|organizacion(es)? de ddhh|madres.*familiares|hijos.*nietos",
    "Comunidades_indigenas": r"indigena|originari|mapuche|comunidad(es)?|pueblo[s]? originario|malon|gaucho",
    "Vecinos": r"vecino|autoconvocad|feriante|rodanter|padre[s]?|madre[s]?|familia",
    "Colectivos_especificos": r"lgbtiq|drag queen|colectivo|mujer|multisectorial de mujer|adolescente|joven",
    "Partidos_politicos": r"partido politico|agrupacion(es)? partidaria|militante[s]?|frente[s]? politico",
    "Publico_general": r"manifestante|sociedad civil|comunidad en general",
}

In [10]:
for var, patron in REGLAS.items():
    df_j[var] = df_j["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)

C:\Users\Federico\AppData\Local\Temp\ipykernel_19800\1998275984.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_j[var] = df_j["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)
C:\Users\Federico\AppData\Local\Temp\ipykernel_19800\1998275984.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_j[var] = df_j["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)
C:\Users\Federico\AppData\Local\Temp\ipykernel_19800\1998275984.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_j[var] = df_j["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)
C:\Users\Federico\AppData\Local\Temp\ipykernel_19800\1998275984.py:2: UserWarning: This pattern is interpreted as a re

In [11]:
cols_actores = list(REGLAS.keys())
print("=" * 60)
print("DISTRIBUCIÓN DE CATEGORÍAS (múltiples por evento)")
print("=" * 60)
for col in cols_actores:
    n = df_j[col].sum()
    pct = df_j[col].mean() * 100
    print(f"{col}: {n} eventos ({pct:.1f}%)")

# Filas sin ninguna categoría
df_j["_tiene_alguno"] = df_j[cols_actores].sum(axis=1)
sin_clasificar = df_j[df_j["_tiene_alguno"] == 0]
print(f"\n⚠ Filas sin clasificar: {len(sin_clasificar)}")
if len(sin_clasificar) > 0:
    print(sin_clasificar[["ID", col_actores]].to_string(index=False))

# Promedio de actores por evento
print(f"\nPromedio de categorías por evento: {df_j['_tiene_alguno'].mean():.2f}")

DISTRIBUCIÓN DE CATEGORÍAS (múltiples por evento)
Docentes: 41 eventos (62.1%)
Gremios_estatales: 13 eventos (19.7%)
Sindicatos_privados: 5 eventos (7.6%)
Organizaciones_sociales: 20 eventos (30.3%)
Comunidades_indigenas: 18 eventos (27.3%)
Vecinos: 11 eventos (16.7%)
Colectivos_especificos: 3 eventos (4.5%)
Partidos_politicos: 1 eventos (1.5%)
Publico_general: 5 eventos (7.6%)

⚠ Filas sin clasificar: 0

Promedio de categorías por evento: 1.77


In [16]:
REGLAS_DEMANDAS = {
    "demanda_salarial": (
        r"salarial|salario|paritaria|aumento|retroactivo|recategoriz|"
        r"recomposicion|sueldo|piso salarial|haberes|deuda.*paritaria|"
        r"deuda.*salarial|incremento salarial|conflicto docente"
    ),
    "demanda_laboral": (
        r"condicion(es)? laboral|convenio colectivo|insalubridad|indumentaria|"
        r"higiene|seguridad laboral|elemento[s]? de|despid|reincorpora|"
        r"contratacion|herramienta[s]?|maquinaria|maltrato|reconocimiento laboral"
    ),
    "demanda_ambiental_territorial": (
        r"mineri|minera|megamineri|zonificacion minera|no a la mina|"
        r"agua|territorio|tierra|desalojo|titularidad|propiedad comunitaria|"
        r"explotacion minera|no a la megamineria"
    ),
    "demanda_politica_legal": (
        r"reforma|constituc|contra.*ley|contra.*gobierno|derogacion|"
        r"veto|ley de pesca|diversificacion productiva|"
        r"contra reforma|prohibicion|restringe"
    ),
    "demanda_ddhh": (
        r"justicia|represion|detencion|femicid|ddhh|derechos humanos|"
        r"repudio|liberacion|detenid|violencia de genero|"
        r"contra las detenciones|apagon del terror|desaparecid"
    ),
    "demanda_servicios": (
        r"luz|agua potable|recoleccion|transporte|energia|electricidad|"
        r"suministro|tarifa|servicio[s]?"
    ),
    "demanda_derechos_especificos": (
        r"lgbtiq|lgbt|drag queen|colectivo|mujer|genero|cannabis|"
        r"pase sanitario|derecho[s]? de las mujeres|diversidad"
    ),
    "demanda_institucional": (
        r"elecciones|renuncia|investigacion|amparo|proceso[s]? judicial|"
        r"carta organica|designacion|asuncion|imputad|"
        r"jueces|magistratura|fiscal"
    ),
    "demanda_conmemoracion": (
        r"conmemoracion|apagon del terror|aniversario|"
        r"memoria|40 marcha|malon de la paz"
    ),
}

In [17]:
def clasificar_demandas(df, col_demanda):
    df["_demanda_norm"] = df[col_demanda].apply(normalizar)
    
    for var, patron in REGLAS_DEMANDAS.items():
        df[var] = df["_demanda_norm"].str.contains(patron, regex=True, na=False).astype(int)
    
    cols = list(REGLAS_DEMANDAS.keys())
    df["_tiene_alguno"] = df[cols].sum(axis=1)
    
    # Reporte
    print("=" * 60)
    print("DISTRIBUCIÓN DE DEMANDAS")
    print("=" * 60)
    for col in cols:
        n = df[col].sum()
        pct = df[col].mean() * 100
        print(f"{col}: {n} ({pct:.1f}%)")
    
    # Sin clasificar (excluir No aplica, Error, No especificado)
    mask_no_esp = df["_demanda_norm"].isin(["no aplica", "error", "no especificado", ""])
    sin_clasificar = df[(df["_tiene_alguno"] == 0) & (~mask_no_esp)]
    print(f"\n⚠ Sin clasificar: {len(sin_clasificar)}")
    if len(sin_clasificar) > 0:
        print(sin_clasificar[[df.columns[0], col_demanda]].head(20).to_string(index=False))
    
    return df.drop(columns=["_demanda_norm", "_tiene_alguno"])

In [18]:
df_j= clasificar_demandas(df_j, "Demanda principal")

DISTRIBUCIÓN DE DEMANDAS
demanda_salarial: 39 (59.1%)
demanda_laboral: 3 (4.5%)
demanda_ambiental_territorial: 2 (3.0%)
demanda_politica_legal: 39 (59.1%)
demanda_ddhh: 8 (12.1%)
demanda_servicios: 0 (0.0%)
demanda_derechos_especificos: 3 (4.5%)
demanda_institucional: 1 (1.5%)
demanda_conmemoracion: 3 (4.5%)

⚠ Sin clasificar: 6
        ID                                                                                  Demanda principal
2023062202                                                                                          sin datos
2023062701                                                                                          sin datos
2023062703                                                                                          sin datos
2023062902                                                                                          sin datos
2023072401                                  no especificado en la nota (corte vinculado al conflicto general)
202307310

C:\Users\Federico\AppData\Local\Temp\ipykernel_19800\358751638.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[var] = df["_demanda_norm"].str.contains(patron, regex=True, na=False).astype(int)


In [20]:
df_j.to_excel("../data/processed/eventos_jujuy_limpio.xlsx", index=False)

In [ ]:
def asignar_chubut(row):
    ta = str(row["tipo_accion"]).strip()
    det = str(row["tipo_accion_detalle"]).lower()
    
    row["tiene_marcha"] = 1 if ta in ["Marcha", "Manifestación", "Caravana"] else 0
    row["tiene_corte"] = 1 if ta == "Corte" else 0
    row["tiene_paro"] = 1 if ta == "Paro" else 0
    row["tiene_acto"] = 1 if ta in ["Acto", "Asamblea"] else 0
    row["tiene_escrache"] = 1 if ta == "Escrache" else 0
    row["tiene_destrozos"] = 1 if ta == "Destrozos" else 0
    row["tiene_toma"] = 1 if ta == "Toma de edificio" else 0
    
    return row

# Jujuy: mapeo con detalle
def asignar_jujuy(row):
    ta = str(row["tipo_accion"]).strip()
    det = str(row["tipo_accion_detalle"]).lower()
    
    row["tiene_marcha"] = 1 if (
        ta in ["Marcha"] or
        (ta == "Mixta" and ("marcha" in det or "movilizacion" in det)) or
        (ta == "Otra" and "marcha" in det)
    ) else 0
    
    row["tiene_corte"] = 1 if ta == "Corte" else 0
    
    row["tiene_paro"] = 1 if (
        (ta == "Otra" and "paro" in det) or
        (ta == "Mixta" and "paro" in det)
    ) else 0
    
    row["tiene_acto"] = 1 if (
        (ta == "Otra" and "asamblea" in det) or
        (ta == "Acampe" and "acampe" in det)
    ) else 0
    
    row["tiene_escrache"] = 0
    row["tiene_destrozos"] = 0
    row["tiene_toma"] = 0
    
    return row

# Aplicar
df = df.apply(lambda row: asignar_chubut(row) if row["provincia"] == "Chubut" else asignar_jujuy(row), axis=1)

In [21]:
# Ver los "Otra"
otras = df_j[df_j["Tipo de acción"] == "Otra"][["ID", "Tipo de acción_detalle", "Observaciones"]]
print("=== JUJUY: Tipo de acción = 'Otra' ===")
print(otras.to_string(index=False))

# Ver los "Mixta"
mixtas = df_j[df_j["Tipo de acción"] == "Mixta"][["ID", "Tipo de acción_detalle", "Observaciones"]]
print("\n=== JUJUY: Tipo de acción = 'Mixta' ===")
print(mixtas.to_string(index=False))

=== JUJUY: Tipo de acción = 'Otra' ===
        ID                                                                                                                                                                                  Tipo de acción_detalle                                                                                                                                                                                                                                                                                                                                                                          Observaciones
2023060302                                                                            Asamblea sindical + emplazamiento a la empresa con amenaza de paro de actividades de 48 horas si no mejora la oferta en 48hs                                                                                                                                                                     